## Remember MNIST from S3 and Run Mock Training

### Objective
Recover the MNIST manifest from S3, remember the single archived dataset object from S3, decode it into tensors, and run a short mock PyTorch training loop.

### Objective
Import the notebook dependencies, configure paths, and define the reusable constants for the example.

In [ ]:
import hashlib
import io
import json

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import laila
from laila.data import S3Pool

POOL_NICKNAME = "mnist_s3_pool"
MANIFEST_NICKNAME = "mnist_s3_manifest"
BATCH_SIZE = 64
MAX_SAMPLES = 512
MAX_STEPS = 5

### Objective
Define a tiny MNIST network and the helper functions used to build the S3 pool, load the manifest, decode the recovered archive, and build a tensor dataset.

In [ ]:
class TinyMNISTNet(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


# These argument names are arbitrary and can be changed by the user.
# laila.args is just a container for user-provided arguments.
# Replace these placeholders with your own AWS credentials before running.
laila.args.AWS_BUCKET_NAME = "your-bucket-name"
laila.args.AWS_ACCESS_KEY_ID = "your-access-key-id"
laila.args.AWS_SECRET_ACCESS_KEY = "your-secret-access-key"
laila.args.AWS_REGION = "us-east-1"


def build_s3_pool() -> S3Pool:
    return S3Pool(
        bucket_name=laila.args.AWS_BUCKET_NAME,
        access_key_id=laila.args.AWS_ACCESS_KEY_ID,
        secret_access_key=laila.args.AWS_SECRET_ACCESS_KEY,
        region_name=laila.args.AWS_REGION,
    )


def manifest_entry_id_from_nickname(manifest_nickname: str) -> str:
    # Nicknames deterministically map to entry IDs, so the manifest can be re-fetched from S3.
    return laila.constant(data=np.zeros(1, dtype=np.uint8), nickname=manifest_nickname).global_id


def entry_payload_to_bytes(payload: object) -> bytes:
    # The upload notebook stored the whole archive as one uint8 array.
    return np.asarray(payload, dtype=np.uint8).tobytes()


def load_manifest_from_s3(manifest_entry_id: str, pool_nickname: str) -> dict:
    with laila.guarantee:
        manifest_entry = laila.remember(manifest_entry_id, pool_nickname=pool_nickname)
    return dict(manifest_entry.data)


def load_mnist_from_npz_bytes(npz_bytes: bytes, max_samples: int) -> tuple[torch.Tensor, torch.Tensor]:
    # Read the compressed MNIST archive directly from memory.
    with np.load(io.BytesIO(npz_bytes)) as data:
        images = data["x_train"][:max_samples].astype(np.float32) / 255.0
        labels = data["y_train"][:max_samples].astype(np.int64)

    image_tensor = torch.from_numpy(images[:, None, :, :])
    label_tensor = torch.from_numpy(labels)
    return image_tensor, label_tensor

### Objective
Register the S3 pool, derive the manifest entry ID from its deterministic nickname, and recover the manifest directly from S3.

In [ ]:
pool_nickname = POOL_NICKNAME
manifest_entry_id = manifest_entry_id_from_nickname(MANIFEST_NICKNAME)

s3_pool = build_s3_pool()
laila.memory.extend(s3_pool, pool_nickname=pool_nickname)

manifest = load_manifest_from_s3(manifest_entry_id, pool_nickname=pool_nickname)

print("Manifest entry id:", manifest_entry_id)
print("Archive entry id:", manifest["entry_id"])
print("Pool nickname:", pool_nickname)

### Objective
Remember the single MNIST archive object from S3, wait on the returned future, and verify that the reconstructed bytes still match the upload-time hash.

In [ ]:
with laila.guarantee:
    recovered_entry = laila.remember(manifest["entry_id"], pool_nickname=pool_nickname)

archive_bytes = entry_payload_to_bytes(recovered_entry.data)
archive_sha256 = hashlib.sha256(archive_bytes).hexdigest()
assert archive_sha256 == manifest["archive_sha256"]

print("Recovered archive size:", len(archive_bytes))
print("Recovered SHA256:", archive_sha256)

### Objective
Decode a small MNIST training subset from the recovered archive and inspect the resulting tensor shapes.

In [ ]:
images, labels = load_mnist_from_npz_bytes(archive_bytes, max_samples=MAX_SAMPLES)

print("Images shape:", tuple(images.shape))
print("Labels shape:", tuple(labels.shape))

### Objective
Create a `DataLoader`, build a tiny classifier, and run a short mock training loop to prove the recovered dataset can feed a normal PyTorch pipeline.

In [ ]:
loader = DataLoader(TensorDataset(images, labels), batch_size=BATCH_SIZE, shuffle=True)
model = TinyMNISTNet()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

model.train()
for step, (batch_images, batch_labels) in enumerate(loader, start=1):
    # Standard training step: forward, loss, backward, optimizer update.
    optimizer.zero_grad()
    logits = model(batch_images)
    loss = criterion(logits, batch_labels)
    loss.backward()
    optimizer.step()

    print(
        f"step={step:02d} "
        f"batch_shape={tuple(batch_images.shape)} "
        f"loss={loss.item():.4f}"
    )
    if step >= MAX_STEPS:
        break

print("Mock MNIST training complete.")

### Objective
Checkpoint the whole run under one **`everything`** complex manifest: the model, the optimizer, a reference to the MNIST data and the config -- memorized to S3 with a single call.

`laila.complex_manifest` is a manifest whose members may be manifests themselves (scope `MANIFEST:COMPLEX`). `Module` / `Optimizer` from `laila.addons.torch` wrap the live objects so that memorizing `everything` snapshots the weights (one detached CPU entry per tensor, plus a weight-free architecture skeleton) and the optimizer state (with a reference to the model each parameter belongs to). The dataset goes in as a *plain* manifest member: it is referenced, never copied and never deleted by this checkpoint.

In [ ]:
from laila.addons.torch import Module, Optimizer

model = Module(model, nickname="mnist_model")
optimizer = Optimizer(optimizer, nickname="mnist_optimizer")

everything = laila.complex_manifest(nickname="mnist_everything")
everything["model"] = model            # -> ModuleManifest (weights + architecture skeleton)
everything["optimizer"] = optimizer    # -> OptimizerManifest (state + model / parameter references)
everything["dataset"] = laila.manifest(  # plain manifest member: shared references, not owned
    data={"manifest": manifest_entry_id, "archive": manifest["entry_id"]},
    nickname="mnist_dataset_refs",
)
everything["config"] = laila.constant(
    data={"batch_size": BATCH_SIZE, "max_samples": MAX_SAMPLES, "max_steps": MAX_STEPS, "lr": 0.1},
    nickname="mnist_config",
)
everything["rng"] = laila.constant(data=torch.get_rng_state(), nickname="mnist_rng")

with laila.guarantee:
    laila.memorize(everything, dst_pool=pool_nickname)

print("Checkpoint:", everything.global_id)
for key in everything.keys():
    print(f"  {key:9s} -> {everything.blueprint[key]}")

### Objective
Resume somewhere else. Drop every local object, remember `everything` by nickname and rebuild it.

When a complex manifest arrives from a pool it goes through **recursive blueprint resolution**: every leaf whose gid carries a `MANIFEST` scope is fetched from the same pool and cast back into its typed manifest (`everything["model"]` is a `ModuleManifest`, `everything["optimizer"]` an `OptimizerManifest`, `everything["dataset"]` the plain `Manifest`), while plain entries stay as gids. `everything.rebuild()` then turns the members into live objects: a `Module` with the architecture and the weights restored, an `Optimizer` bound to that model's parameters with its state loaded, and the config / RNG entries.

In [ ]:
reference_state = {name: tensor.clone() for name, tensor in model.state_dict().items()}
del model, optimizer, everything

with laila.guarantee:
    ref = laila.remember("MANIFEST:COMPLEX:mnist_everything", dst_pool=pool_nickname)
everything = ref.result

print("Resolved members:", {key: type(value).__name__ for key, value in everything.members.items()})

live = everything.rebuild()
model, optimizer = live["model"], live["optimizer"]
print("Rebuilt:", {key: type(value).__name__ for key, value in live.items()})

assert all(torch.equal(reference_state[name], tensor) for name, tensor in model.state_dict().items())
assert all(id(p) in {id(q) for q in model.parameters()} for g in optimizer.param_groups for p in g["params"])
torch.set_rng_state(live["rng"].data)
print("Config:", live["config"].data)
print("Dataset references:", live["dataset"].blueprint)

# Training continues right where it stopped.
model.train()
batch_images, batch_labels = next(iter(loader))
optimizer.zero_grad()
loss = criterion(model(batch_images), batch_labels)
loss.backward()
optimizer.step()
print(f"resumed step loss={loss.item():.4f}")

### Objective
Remove the checkpoint. Forgetting `everything` deletes what it owns -- the model and optimizer checkpoints (every tensor), the config and RNG entries, and the blueprint records -- while the MNIST archive referenced through the plain dataset manifest is left untouched.

In [ ]:
with laila.guarantee:
    laila.forget(everything, pool=pool_nickname)

print("Checkpoint removed; the MNIST archive is still in S3.")